In [18]:
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
prices = pd.read_csv("../data/processed/dk2_price_features_next_hour.csv")
weather = pd.read_csv("../data/raw/weather_copenhagen_2024_01_01_2025_02_19.csv")
prod = pd.read_csv("../data/processed/dk2_production_consumption_clean.csv")

In [20]:
prices["HourUTC"] = pd.to_datetime(prices["HourUTC"])
prices["HourDK"] = pd.to_datetime(prices["HourDK"])
prod["HourUTC"] = pd.to_datetime(prod["HourUTC"])
weather["HourUTC"] = pd.to_datetime(weather["HourUTC"])

In [21]:
print("Duplicated prices HourDK:", prices["HourUTC"].duplicated().sum())
print("Duplicated weather time:", weather["HourUTC"].duplicated().sum())
print("Duplicated prod HourDK:", prod["HourUTC"].duplicated().sum())

Duplicated prices HourDK: 0
Duplicated weather time: 0
Duplicated prod HourDK: 0


In [22]:
prices = prices.sort_values("HourUTC").reset_index(drop=True)
weather = weather.sort_values("HourUTC").reset_index(drop=True)
prod = prod.sort_values("HourUTC").reset_index(drop=True)

In [23]:
df = prices.merge(
    weather,
    on="HourUTC",
    how="inner"
)

df = df.merge(
    prod,
    on=["HourUTC", "PriceArea"],
    how="inner"
)

In [24]:
df = df.rename(columns={"HourDK_x": "HourDK"})
df = df.drop(columns=["HourDK_y"])

In [25]:
print("Rows:", len(df))
print("Date range:", df["HourUTC"].min(), "→", df["HourUTC"].max())
print(df.isna().sum())

Rows: 9815
Date range: 2024-01-07 23:00:00 → 2025-02-19 21:00:00
HourUTC                     0
HourDK                      0
PriceArea                   0
SpotPriceEUR                0
hour                        0
day_of_week                 0
month                       0
year                        0
is_weekend                  0
price_lag_24h               0
price_lag_48h               0
price_lag_168h              0
price_rolling_mean_24h      0
price_rolling_std_24h       0
price_rolling_mean_168h     0
price_rolling_std_168h      0
target_next_hour            0
temperature_2m              0
wind_speed_10m              0
wind_speed_100m             0
cloud_cover                 0
shortwave_radiation         0
GrossConsumptionMWh         0
CentralPowerMWh             0
LocalPowerMWh               0
CommercialPowerMWh          0
offshore_wind_mwh           0
onshore_wind_mwh            0
solar_mwh                   0
total_wind_mwh              0
renewable_generation_mwh    0
net_l

In [26]:
df.columns.tolist()

['HourUTC',
 'HourDK',
 'PriceArea',
 'SpotPriceEUR',
 'hour',
 'day_of_week',
 'month',
 'year',
 'is_weekend',
 'price_lag_24h',
 'price_lag_48h',
 'price_lag_168h',
 'price_rolling_mean_24h',
 'price_rolling_std_24h',
 'price_rolling_mean_168h',
 'price_rolling_std_168h',
 'target_next_hour',
 'temperature_2m',
 'wind_speed_10m',
 'wind_speed_100m',
 'cloud_cover',
 'shortwave_radiation',
 'GrossConsumptionMWh',
 'CentralPowerMWh',
 'LocalPowerMWh',
 'CommercialPowerMWh',
 'offshore_wind_mwh',
 'onshore_wind_mwh',
 'solar_mwh',
 'total_wind_mwh',
 'renewable_generation_mwh',
 'net_load_mwh',
 'ExchangeSE_MWh',
 'ExchangeGE_MWh',
 'ExchangeGreatBelt_MWh',
 'PowerToHeatMWh']

In [27]:
print("Rows:", len(df))
print("Date range UTC:", df["HourUTC"].min(), "→", df["HourUTC"].max())
print(df.isna().sum())

Rows: 9815
Date range UTC: 2024-01-07 23:00:00 → 2025-02-19 21:00:00
HourUTC                     0
HourDK                      0
PriceArea                   0
SpotPriceEUR                0
hour                        0
day_of_week                 0
month                       0
year                        0
is_weekend                  0
price_lag_24h               0
price_lag_48h               0
price_lag_168h              0
price_rolling_mean_24h      0
price_rolling_std_24h       0
price_rolling_mean_168h     0
price_rolling_std_168h      0
target_next_hour            0
temperature_2m              0
wind_speed_10m              0
wind_speed_100m             0
cloud_cover                 0
shortwave_radiation         0
GrossConsumptionMWh         0
CentralPowerMWh             0
LocalPowerMWh               0
CommercialPowerMWh          0
offshore_wind_mwh           0
onshore_wind_mwh            0
solar_mwh                   0
total_wind_mwh              0
renewable_generation_mwh    0
n

In [28]:
df.to_csv(
    "../data/processed/dk2_merged_price_weather_energy.csv",
    index=False
)

In [29]:
price_hours = set(prices["HourUTC"])
weather_hours = set(weather["HourUTC"])
prod_hours = set(prod["HourUTC"])

common_hours = price_hours & weather_hours & prod_hours

print("Prices rows:", len(price_hours))
print("Weather rows:", len(weather_hours))
print("Prod rows:", len(prod_hours))
print("Common rows:", len(common_hours))

print("Missing in prices:", sorted((weather_hours & prod_hours) - price_hours)[-5:])
print("Missing in weather:", sorted((price_hours & prod_hours) - weather_hours)[-5:])
print("Missing in prod:", sorted((price_hours & weather_hours) - prod_hours)[-5:])

Prices rows: 9815
Weather rows: 9984
Prod rows: 9984
Common rows: 9815
Missing in prices: [Timestamp('2024-01-07 19:00:00'), Timestamp('2024-01-07 20:00:00'), Timestamp('2024-01-07 21:00:00'), Timestamp('2024-01-07 22:00:00'), Timestamp('2025-02-19 22:00:00')]
Missing in weather: []
Missing in prod: []
